# 01 — Leituras perdidas e autoencoders

Uma usina registra, de hora em hora, as condições do ambiente e o funcionamento de uma turbina a gás.
Sensores falham, entram em manutenção ou perdem registros. Queremos investigar se é possível estimar
uma leitura perdida a partir das **outras leituras do mesmo registro**, e se um autoencoder é uma
boa ferramenta para isso.

In [ ]:
from pathlib import Path
import sys

# Raiz do projeto: a primeira pasta, a partir da atual, que contém src/dados.py.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dados.py").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

In [ ]:
import json

import numpy as np
import pandas as pd

from src.dados import COLUNAS, DESCRICAO, PASTA_ARTEFATOS, carregar_conjunto
from src import visual_utils as vis

## 1.1 — Sensores de uma turbina a gás e leituras perdidas

> **Pergunta:** como podemos recuperar uma leitura que deixou de ser registrada?

### O que representa cada variável

In [ ]:
pd.DataFrame(DESCRICAO, index=["descrição", "unidade"]).T

Toda a exploração deste notebook usa **somente o treino (2011–2012)**. Validação e teste ficam
guardados para as comparações e para a avaliação final.

In [ ]:
leituras.describe().T[["mean", "std", "min", "50%", "max"]].round(2)

As escalas são muito diferentes: a pressão ambiente varia poucos mbar em torno de 1013, enquanto a
temperatura de entrada da turbina passa de 1000 °C. Um erro de 1 unidade não significa a mesma coisa
em cada sensor. O CO chama atenção: mediana perto de 1,4 mg/m³ e máximo acima de 44.

### Quais sensores têm distribuições diferentes?

In [ ]:
vis.distribuicao_original_e_log(leituras["CO"], "CO", "mg/m³").show()

A distribuição do CO tem cauda longa: poucos registros com valores muito altos. Depois de
`log1p` (logaritmo de 1 + x), a distribuição fica bem mais concentrada. Essa transformação será
usada na preparação dos dados para a rede.

### Quais leituras parecem relacionadas?

In [ ]:
correlacao = leituras.corr()
vis.mapa_correlacao(leituras).show()

In [ ]:
unidades = {coluna: unidade for coluna, (_, unidade) in DESCRICAO.items()}
vis.dispersao_pares(leituras, [("TEY", "CDP"), ("TEY", "AP")], unidades).show()

Energia gerada (TEY), pressão do compressor (CDP), pressão de exaustão (GTEP) e temperatura de entrada
(TIT) andam juntas, com correlações próximas de 0,9 ou mais. Já a pressão ambiente (AP) quase não
acompanha a operação da turbina.

### Por que alguns sensores devem ser mais difíceis de recuperar?

In [ ]:
sem_diagonal = correlacao.abs().where(~np.eye(len(COLUNAS), dtype=bool))
maior_relacao = sem_diagonal.max().sort_values()
maior_relacao.round(2)

Para cada sensor, este é o maior valor absoluto de correlação com qualquer outro sensor.
AP, AH, AT e NOX têm no máximo uma relação moderada com os demais: quando uma dessas leituras
faltar, haverá menos informação no restante do registro para estimá-la. A correlação mede só
relações lineares entre pares, então é uma pista, não uma garantia.

### Como esconder valores para testar a recuperação

A base original não tem leituras ausentes. Para avaliar um método, escondemos leituras de registros
completos, pedimos ao método que as recupere e comparamos com o valor real, que conhecemos.

Uma primeira forma de preencher as lacunas: usar a mediana de cada sensor no treino. O erro é
medido **somente nas posições escondidas**.

In [ ]:
preenchida = amostra_incompleta.fillna(leituras.median())
erros = (preenchida - amostra).where(escondido)

erro_escondido = erros.stack()
rmse = np.sqrt((erro_escondido ** 2).mean())
mae = erro_escondido.abs().mean()
print(f"{escondido.sum()} leituras escondidas | RMSE = {rmse:.2f} | MAE = {mae:.2f}  (unidades misturadas)")
erros.round(2)

**RMSE** (raiz do erro quadrático médio) pesa mais os erros grandes; **MAE** (erro absoluto médio)
é a média do tamanho dos erros. O número acima mistura °C, mbar e mg/m³, então não dá para
interpretá-lo. Por isso as métricas gerais serão calculadas na **escala padronizada** (cada sensor
dividido pelo seu desvio-padrão), e as unidades originais serão usadas sensor por sensor.

### Treino, validação e teste

In [ ]:
manifesto = json.loads((PASTA_ARTEFATOS / "manifesto_dados.json").read_text(encoding="utf-8"))
papel = {"treino": "aprender e ajustar o pré-processamento",
         "validacao": "comparar configurações e escolher",
         "teste": "avaliação final"}
pd.DataFrame({"anos": manifesto["anos_por_conjunto"], "registros": manifesto["registros_por_conjunto"],
              "papel": papel})

Se usássemos os mesmos dados para treinar, escolher e avaliar, o resultado final seria otimista.
A validação recebe todas as decisões; o teste, com anos que nenhum modelo viu, só mede o resultado
no fim.

Os registros estão em ordem cronológica dentro de cada ano, mas não têm o horário de cada leitura,
e cada ano tem menos registros (7.152 a 7.628) do que horas (8.760). Por isso a unidade de análise
é **um registro com as 11 leituras da mesma hora**, e não uma sequência no tempo. Dividir por ano,
em vez de sortear registros, também mostra se o modelo funciona em anos que não viu.

**Conclusão:** algumas leituras possuem relação forte com outras, mas isso não significa que todas
possam ser estimadas com precisão. Sensores pouco relacionados com o restante do registro devem ser
os mais difíceis de recuperar.

## 1.2 — Autoencoders: encoder, decoder e espaço latente

> **Pergunta:** por que aprender a reconstruir um registro pode ajudar a recuperar informações?

### Encoder e decoder

Muitos modelos de redes neurais têm duas partes. O **encoder** resume a entrada em uma
representação interna; o **decoder** gera a saída a partir dessa representação. Entrada e saída
podem ser bem diferentes: uma frase em português e sua tradução, uma imagem e sua legenda, uma foto
e o mapa que separa os objetos dela.

### Autoencoder: a saída é a própria entrada

Um autoencoder é o caso em que a saída desejada é **o próprio registro de entrada**, depois de
passar por um gargalo:

```
 registro x            ENCODER              vetor latente z          DECODER           registro x̂
 11 leituras   ──►   11 → 32 → 2    ──►     2 números       ──►   2 → 32 → 11   ──►   11 leituras

 erro de reconstrução = média de (x − x̂)²
```

- **Representação latente (z):** o resumo do registro, com menos números que a entrada.
- **Gargalo:** como z é pequeno, a rede não consegue simplesmente copiar as 11 leituras; para
  reconstruí-las, precisa representar as relações entre elas.
- **Erro de reconstrução:** a diferença entre a entrada e a saída. É o que o treino minimiza.

### Um resumo com menos números que o registro

Três sensores (TEY, CDP e GTEP) variam quase juntos. Se guardarmos **um único número** por
registro, a média das três leituras padronizadas, conseguimos reconstruir cada uma delas? E se
colocarmos no mesmo resumo a pressão ambiente (AP), que quase não acompanha a operação?

In [ ]:
padronizado = (leituras - leituras.mean()) / leituras.std()

def erro_do_resumo(sensores):
    resumo = padronizado[sensores].mean(axis=1)        # vários sensores -> 1 número
    # reconstrução: cada sensor recebe o próprio resumo de volta
    return {s: float(((padronizado[s] - resumo) ** 2).mean()) for s in sensores}

cenarios = {"TEY, CDP e GTEP": ["TEY", "CDP", "GTEP"], "TEY, CDP, GTEP e AP": ["TEY", "CDP", "GTEP", "AP"]}
exemplo = pd.DataFrame([{"resumo de": nome, "sensor": s, "erro": e}
                        for nome, sensores in cenarios.items() for s, e in erro_do_resumo(sensores).items()])
exemplo.pivot(index="sensor", columns="resumo de", values="erro").round(3)

In [ ]:
vis.barras_agrupadas(exemplo, "sensor", "erro", "resumo de",
                     "Erro ao reconstruir cada sensor a partir de um único número (treino)",
                     "erro quadrático médio (escala padronizada)",
                     cores={"TEY, CDP e GTEP": vis.AZUL_ESCURO, "TEY, CDP, GTEP e AP": vis.CONTRASTE}).show()

Com os três sensores de operação, um único número reconstrói cada um deles com erro entre 0,003 e
0,014 (na escala padronizada, prever sempre a média dá erro 1). Quando a pressão ambiente entra no
mesmo resumo, o erro dela fica perto de 1, como se nada tivesse sido guardado sobre ela, e o resumo
ainda piora a reconstrução dos outros três (de 0,11 a 0,14). Um resumo pequeno funciona para
leituras que variam juntas.

Um autoencoder aprende esse tipo de resumo sozinho: com mais de uma dimensão latente, com camadas
não lineares e com pesos ajustados por gradiente, em vez de uma média escolhida à mão. Se o resumo
capturar as relações entre as leituras, as leituras presentes podem ajudar a estimar a que faltou.

**Conclusão:** o autoencoder comprime o registro em um espaço latente e o reconstrói. O gargalo
obriga a rede a representar as relações entre as leituras, e são essas relações que podem preencher
lacunas. Uma questão que permanece é como construir e treinar essa rede em PyTorch.